In [5]:
# =========================================
# Import packages and set up Neo4j
# =========================================
from dotenv import load_dotenv
import os

# Common data processing
import json
import textwrap

# Langchain (modern)
from langchain_community.graphs import Neo4jGraph
from langchain_community.vectorstores import Neo4jVector
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter

# LCEL (modern chains)
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

# Warning control
import warnings
warnings.filterwarnings("ignore")

In [6]:
# Load from environment
load_dotenv('.env', override=True)
NEO4J_URI = os.getenv('NEO4J_URI')
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')
NEO4J_DATABASE = os.getenv('NEO4J_DATABASE') or 'neo4j'
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

In [7]:
# Initialize models
embeddings = OpenAIEmbeddings()
llm = ChatOpenAI(temperature=0)

In [8]:
# Constants
VECTOR_INDEX_NAME = 'form_10k_chunks'
VECTOR_NODE_LABEL = 'Chunk'
VECTOR_SOURCE_PROPERTY = 'text'
VECTOR_EMBEDDING_PROPERTY = 'textEmbedding'

In [10]:
# Neo4j connection
kg = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE
)

In [12]:
# =========================================
# Take a look at a Form 10-K json file
# =========================================
first_file_name = "./data/form10k/0000950170-23-027948.json"
first_file_as_object = json.load(open(first_file_name))

print(type(first_file_as_object))
for k, v in first_file_as_object.items():
    print(k, type(v))

item1_text = first_file_as_object['item1']
print(item1_text[:1500])

<class 'dict'>
item1 <class 'str'>
item1a <class 'str'>
item7 <class 'str'>
item7a <class 'str'>
cik <class 'str'>
cusip6 <class 'str'>
cusip <class 'list'>
names <class 'list'>
source <class 'str'>
>Item 1.  
Business


Overview


NetApp, Inc. (NetApp, we, us or the Company) is a global cloud-led, data-centric software company. We were incorporated in 1992 and are headquartered in San Jose, California. Building on more than three decades of innovation, we give customers the freedom to manage applications and data across hybrid multicloud environments. Our portfolio of cloud services, and storage infrastructure, powered by intelligent data management software, enables applications to run faster, more reliably, and more securely, all at a lower cost.


Our opportunity is defined by the durable megatrends of data-driven digital and cloud transformations. NetApp helps organizations meet the complexities created by rapid data and cloud growth, multi-cloud management, and the adoption of ne

In [13]:
# =========================================
# Split Form 10-K sections into chunks
# =========================================
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=200,
    length_function=len,
    is_separator_regex=False,
)

item1_text_chunks = text_splitter.split_text(item1_text)

print(type(item1_text_chunks))
print(len(item1_text_chunks))
print(item1_text_chunks[0])

<class 'list'>
254
>Item 1.  
Business


Overview


NetApp, Inc. (NetApp, we, us or the Company) is a global cloud-led, data-centric software company. We were incorporated in 1992 and are headquartered in San Jose, California. Building on more than three decades of innovation, we give customers the freedom to manage applications and data across hybrid multicloud environments. Our portfolio of cloud services, and storage infrastructure, powered by intelligent data management software, enables applications to run faster, more reliably, and more securely, all at a lower cost.


Our opportunity is defined by the durable megatrends of data-driven digital and cloud transformations. NetApp helps organizations meet the complexities created by rapid data and cloud growth, multi-cloud management, and the adoption of next-generation technologies, such as AI, Kubernetes, and modern databases. Our modern approach to hybrid, multicloud infrastructure and data management, which we term ‘evolved cloud

In [14]:
def split_form10k_data_from_file(file):
    chunks_with_metadata = []
    file_as_object = json.load(open(file))

    for item in ['item1', 'item1a', 'item7', 'item7a']:
        print(f'Processing {item} from {file}')
        item_text = file_as_object[item]
        item_text_chunks = text_splitter.split_text(item_text)

        chunk_seq_id = 0
        for chunk in item_text_chunks[:20]:
            form_id = file[file.rindex('/') + 1:file.rindex('.')]

            chunks_with_metadata.append({
                'text': chunk,
                'f10kItem': item,
                'chunkSeqId': chunk_seq_id,
                'formId': f'{form_id}',
                'chunkId': f'{form_id}-{item}-chunk{chunk_seq_id:04d}',
                'names': file_as_object['names'],
                'cik': file_as_object['cik'],
                'cusip6': file_as_object['cusip6'],
                'source': file_as_object['source'],
            })
            chunk_seq_id += 1

        print(f'\tSplit into {chunk_seq_id} chunks')

    return chunks_with_metadata


first_file_chunks = split_form10k_data_from_file(first_file_name)
print(first_file_chunks[0])

Processing item1 from ./data/form10k/0000950170-23-027948.json
	Split into 20 chunks
Processing item1a from ./data/form10k/0000950170-23-027948.json
	Split into 1 chunks
Processing item7 from ./data/form10k/0000950170-23-027948.json
	Split into 1 chunks
Processing item7a from ./data/form10k/0000950170-23-027948.json
	Split into 1 chunks
{'text': '>Item 1.  \nBusiness\n\n\nOverview\n\n\nNetApp, Inc. (NetApp, we, us or the Company) is a global cloud-led, data-centric software company. We were incorporated in 1992 and are headquartered in San Jose, California. Building on more than three decades of innovation, we give customers the freedom to manage applications and data across hybrid multicloud environments. Our portfolio of cloud services, and storage infrastructure, powered by intelligent data management software, enables applications to run faster, more reliably, and more securely, all at a lower cost.\n\n\nOur opportunity is defined by the durable megatrends of data-driven digital an

In [15]:
# =========================================
# Create graph nodes using text chunks
# =========================================
merge_chunk_node_query = """
MERGE (c:Chunk {chunkId: $chunkParam.chunkId})
ON CREATE SET 
    c.names = $chunkParam.names,
    c.formId = $chunkParam.formId,
    c.cik = $chunkParam.cik,
    c.cusip6 = $chunkParam.cusip6,
    c.source = $chunkParam.source,
    c.f10kItem = $chunkParam.f10kItem,
    c.chunkSeqId = $chunkParam.chunkSeqId,
    c.text = $chunkParam.text
RETURN c
"""

kg.query("""
CREATE CONSTRAINT unique_chunk IF NOT EXISTS
FOR (c:Chunk) REQUIRE c.chunkId IS UNIQUE
""")

node_count = 0
for chunk in first_file_chunks:
    kg.query(merge_chunk_node_query, params={'chunkParam': chunk})
    node_count += 1

print(f"Created {node_count} nodes")

print(kg.query("MATCH (n) RETURN count(n) AS nodeCount"))

Created 23 nodes
[{'nodeCount': 195}]


In [16]:
# =========================================
# Create a vector index
# =========================================
kg.query(f"""
CREATE VECTOR INDEX {VECTOR_INDEX_NAME} IF NOT EXISTS
FOR (c:{VECTOR_NODE_LABEL}) ON (c.{VECTOR_EMBEDDING_PROPERTY})
OPTIONS {{
  indexConfig: {{
    `vector.dimensions`: 1536,
    `vector.similarity_function`: 'cosine'
  }}
}}
""")

print(kg.query("SHOW INDEXES"))

[{'id': 6, 'name': 'form_10k_chunks', 'state': 'ONLINE', 'populationPercent': 100.0, 'type': 'VECTOR', 'entityType': 'NODE', 'labelsOrTypes': ['Chunk'], 'properties': ['textEmbedding'], 'indexProvider': 'vector-3.0', 'owningConstraint': None, 'lastRead': None, 'readCount': None}, {'id': 1, 'name': 'index_343aff4e', 'state': 'ONLINE', 'populationPercent': 100.0, 'type': 'LOOKUP', 'entityType': 'NODE', 'labelsOrTypes': None, 'properties': None, 'indexProvider': 'token-lookup-1.0', 'owningConstraint': None, 'lastRead': neo4j.time.DateTime(2026, 3, 28, 12, 47, 24, 708000000, tzinfo=<UTC>), 'readCount': 172}, {'id': 2, 'name': 'index_f7700477', 'state': 'ONLINE', 'populationPercent': 100.0, 'type': 'LOOKUP', 'entityType': 'RELATIONSHIP', 'labelsOrTypes': None, 'properties': None, 'indexProvider': 'token-lookup-1.0', 'owningConstraint': None, 'lastRead': neo4j.time.DateTime(2024, 1, 18, 11, 46, 2, 950000000, tzinfo=<UTC>), 'readCount': 1}, {'id': 3, 'name': 'movie_tagline_embeddings', 'state

In [17]:
# =========================================
# Calculate embedding vectors for chunks and populate index
# =========================================
chunks = kg.query("""
MATCH (c:Chunk)
WHERE c.textEmbedding IS NULL
RETURN elementId(c) AS id, c.text AS text
""")

for chunk in chunks:
    chunk["embedding"] = embeddings.embed_query(chunk["text"])

kg.query("""
UNWIND $rows AS row
MATCH (c:Chunk)
WHERE elementId(c) = row.id
SET c.textEmbedding = row.embedding
""", params={"rows": chunks})

kg.refresh_schema()
print(kg.schema)

Node properties:
Movie {title: STRING, taglineEmbedding: LIST, tagline: STRING, released: INTEGER}
Person {born: INTEGER, name: STRING, lastSeen: INTEGER}
Chunk {textEmbedding: LIST, chunkId: STRING, f10kItem: STRING, chunkSeqId: INTEGER, text: STRING, cik: STRING, cusip6: STRING, names: LIST, formId: STRING, source: STRING}
Relationship properties:
ACTED_IN {roles: LIST}
REVIEWED {summary: STRING, rating: INTEGER}
The relationships:
(:Person)-[:ACTED_IN]->(:Movie)
(:Person)-[:DIRECTED]->(:Movie)
(:Person)-[:PRODUCED]->(:Movie)
(:Person)-[:WROTE]->(:Movie)
(:Person)-[:FOLLOWS]->(:Person)
(:Person)-[:REVIEWED]->(:Movie)
(:Person)-[:KNOWS]->(:Person)


In [18]:
# =========================================
# Use similarity search to find relevant chunks
# =========================================
def neo4j_vector_search(question):
    embedding = embeddings.embed_query(question)

    query = """
    CALL db.index.vector.queryNodes($index_name, $top_k, $embedding)
    YIELD node, score
    RETURN score, node.text AS text
    """

    return kg.query(query, params={
        "index_name": VECTOR_INDEX_NAME,
        "top_k": 5,
        "embedding": embedding
    })


search_results = neo4j_vector_search(
    "In a single sentence, tell me about Netapp."
)

print(search_results[0])

{'score': 0.9307861328125, 'text': '>Item 1.  \nBusiness\n\n\nOverview\n\n\nNetApp, Inc. (NetApp, we, us or the Company) is a global cloud-led, data-centric software company. We were incorporated in 1992 and are headquartered in San Jose, California. Building on more than three decades of innovation, we give customers the freedom to manage applications and data across hybrid multicloud environments. Our portfolio of cloud services, and storage infrastructure, powered by intelligent data management software, enables applications to run faster, more reliably, and more securely, all at a lower cost.\n\n\nOur opportunity is defined by the durable megatrends of data-driven digital and cloud transformations. NetApp helps organizations meet the complexities created by rapid data and cloud growth, multi-cloud management, and the adoption of next-generation technologies, such as AI, Kubernetes, and modern databases. Our modern approach to hybrid, multicloud infrastructure and data management, w

In [19]:
# =========================================
# Set up a LangChain RAG workflow to chat with the form
# =========================================

# Vector store
neo4j_vector_store = Neo4jVector.from_existing_graph(
    embedding=embeddings,
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    index_name=VECTOR_INDEX_NAME,
    node_label=VECTOR_NODE_LABEL,
    text_node_properties=[VECTOR_SOURCE_PROPERTY],
    embedding_node_property=VECTOR_EMBEDDING_PROPERTY,
)

retriever = neo4j_vector_store.as_retriever()

In [20]:
# Prompt
prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the context below.

Context:
{context}

Question:
{question}

Answer:
""")

In [21]:
# LCEL RAG chain
rag_chain = (
    {
        "context": retriever,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [22]:
# =========================================
# Helper function
# =========================================
def prettychain(question: str):
    response = rag_chain.invoke(question)
    print(textwrap.fill(response, 80))

In [23]:
# =========================================
# Queries
# =========================================
prettychain("What is Netapp's primary business?")

Netapp's primary business is focused on enterprise storage and data management,
cloud storage, and cloud operations markets.


In [24]:
prettychain("Where is Netapp headquartered?")

Netapp is headquartered in San Jose, California.


In [25]:
prettychain("""
Tell me about Netapp. 
Limit your answer to a single sentence.
""")

NetApp, Inc. is a global cloud-led, data-centric software company that provides
customers with the freedom to manage applications and data across hybrid
multicloud environments.


In [26]:
prettychain("""
Tell me about Apple. 
Limit your answer to a single sentence.
""")

The context provided does not mention Apple, so there is no information
available to provide a sentence about Apple.


In [27]:
prettychain("""
Tell me about Apple. 
Limit your answer to a single sentence.
If you are unsure about the answer, say you don't know.
""")

I don't know.


In [1]:
prettychain("Tell me about Netapp?")

NameError: name 'prettychain' is not defined